# 3. Model Training & Evaluation
**AAI-540 MLOps Project — Group 6: Student Performance Prediction**

This notebook covers:
- Train/Validation/Test split (70/15/15)
- Training 3 models: Logistic Regression, Decision Tree, Random Forest
- Hyperparameter tuning
- Model comparison and evaluation
- Feature importance analysis
- SageMaker Model Registry

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score, roc_curve)
import joblib
import sagemaker
import boto3
import os
import tarfile
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

session = sagemaker.Session()
bucket = session.default_bucket()
prefix = 'student-performance'
role = sagemaker.get_execution_role()
RANDOM_STATE = 42

## 3.1 Load Data & Split

In [ ]:
df = pd.read_csv('../data/processed/student_processed.csv')
X = df.drop(columns=['performance'])
y = df['performance']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)

print(f'Training:   {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Validation: {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Test:       {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.0f}%)')

In [ ]:
# Save splits
pd.concat([X_train, y_train], axis=1).to_csv('../data/processed/train.csv', index=False)
pd.concat([X_val, y_val], axis=1).to_csv('../data/processed/validation.csv', index=False)
pd.concat([X_test, y_test], axis=1).to_csv('../data/processed/test.csv', index=False)

# Upload to S3
for f in ['train.csv', 'validation.csv', 'test.csv']:
    session.upload_data(path=f'../data/processed/{f}', bucket=bucket, key_prefix=f'{prefix}/processed')
print('Split data saved and uploaded to S3')

## 3.2 Model Training
### Logistic Regression

In [ ]:
lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_val_pred = lr_model.predict(X_val)

print('=== Logistic Regression ===')
print(f'Training Accuracy:   {accuracy_score(y_train, lr_model.predict(X_train)):.4f}')
print(f'Validation Accuracy: {accuracy_score(y_val, lr_val_pred):.4f}')
print(classification_report(y_val, lr_val_pred, target_names=['Low', 'High']))

### Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)
dt_val_pred = dt_model.predict(X_val)

print('=== Decision Tree ===')
print(f'Training Accuracy:   {accuracy_score(y_train, dt_model.predict(X_train)):.4f}')
print(f'Validation Accuracy: {accuracy_score(y_val, dt_val_pred):.4f}')
print(classification_report(y_val, dt_val_pred, target_names=['Low', 'High']))

### Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_model.fit(X_train, y_train)
rf_val_pred = rf_model.predict(X_val)

print('=== Random Forest ===')
print(f'Training Accuracy:   {accuracy_score(y_train, rf_model.predict(X_train)):.4f}')
print(f'Validation Accuracy: {accuracy_score(y_val, rf_val_pred):.4f}')
print(classification_report(y_val, rf_val_pred, target_names=['Low', 'High']))

## 3.3 Model Comparison

In [ ]:
models = {'Logistic Regression': (lr_model, lr_val_pred),
          'Decision Tree': (dt_model, dt_val_pred),
          'Random Forest': (rf_model, rf_val_pred)}

results = []
for name, (model, pred) in models.items():
    results.append({'Model': name,
        'Accuracy': accuracy_score(y_val, pred),
        'Precision': precision_score(y_val, pred),
        'Recall': recall_score(y_val, pred),
        'F1-Score': f1_score(y_val, pred),
        'AUC-ROC': roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])})

results_df = pd.DataFrame(results).set_index('Model')
results_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
results_df.plot(kind='bar', ax=ax, colormap='viridis', edgecolor='black', alpha=0.8)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison (Validation Set)')
ax.set_ylim(0, 1.1)
plt.xticks(rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

## 3.4 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, (model, pred)) in enumerate(models.items()):
    cm = confusion_matrix(y_val, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
    axes[i].set_title(f'{name}\nAcc: {accuracy_score(y_val, pred):.3f}')
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')
plt.suptitle('Confusion Matrices (Validation Set)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 3.5 ROC Curves

In [ ]:
plt.figure(figsize=(10, 7))
colors = ['#3498db', '#e74c3c', '#2ecc71']
for (name, (model, _)), color in zip(models.items(), colors):
    y_proba = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    auc = roc_auc_score(y_val, y_proba)
    plt.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 3.6 Hyperparameter Tuning — Random Forest

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV F1-Score: {grid_search.best_score_:.4f}')

In [ ]:
best_rf = grid_search.best_estimator_
best_pred = best_rf.predict(X_val)

print('=== Tuned Random Forest (Validation) ===')
print(f'Accuracy:  {accuracy_score(y_val, best_pred):.4f}')
print(f'F1-Score:  {f1_score(y_val, best_pred):.4f}')
print(f'AUC-ROC:   {roc_auc_score(y_val, best_rf.predict_proba(X_val)[:, 1]):.4f}')
print(classification_report(y_val, best_pred, target_names=['Low', 'High']))

## 3.7 Cross-Validation

In [ ]:
print('=== 5-Fold Cross-Validation ===')
for name, model in [('Logistic Regression', lr_model), ('Decision Tree', dt_model), ('Random Forest (Tuned)', best_rf)]:
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    print(f'{name:30s} F1: {cv.mean():.4f} (+/- {cv.std():.4f})')

## 3.8 Feature Importance

In [ ]:
feat_imp = pd.DataFrame({'feature': X.columns, 'importance': best_rf.feature_importances_}
                        ).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='importance', y='feature', data=feat_imp.head(20), palette='viridis', edgecolor='black')
plt.title('Top 20 Feature Importances (Tuned Random Forest)')
plt.tight_layout()
plt.show()

print('Top 10 Features:')
print(feat_imp.head(10).to_string(index=False))

## 3.9 Final Test Set Evaluation

In [ ]:
test_pred = best_rf.predict(X_test)

print('=== Final Test Evaluation (Tuned Random Forest) ===')
print(f'Accuracy:  {accuracy_score(y_test, test_pred):.4f}')
print(f'F1-Score:  {f1_score(y_test, test_pred):.4f}')
print(f'AUC-ROC:   {roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1]):.4f}')
print(classification_report(y_test, test_pred, target_names=['Low', 'High']))

plt.figure(figsize=(7, 5))
sns.heatmap(confusion_matrix(y_test, test_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
plt.title('Test Set — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 3.10 Save Models & Register in SageMaker

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(lr_model, '../models/logistic_regression.joblib')
joblib.dump(dt_model, '../models/decision_tree.joblib')
joblib.dump(best_rf, '../models/random_forest_best.joblib')

# Create model.tar.gz for SageMaker
with tarfile.open('../models/model.tar.gz', 'w:gz') as tar:
    tar.add('../models/random_forest_best.joblib', arcname='model.joblib')

# Upload to S3
model_artifact = session.upload_data(
    path='../models/model.tar.gz',
    bucket=bucket,
    key_prefix=f'{prefix}/model'
)
print(f'Model artifact uploaded to: {model_artifact}')

In [ ]:
# Register model in SageMaker Model Registry
from sagemaker.sklearn import SKLearnModel

sklearn_model = SKLearnModel(
    model_data=model_artifact,
    role=role,
    framework_version='1.2-1',
    entry_point='inference.py',
    source_dir='../src/',
    sagemaker_session=session
)

model_package_group_name = 'StudentPerformanceModelGroup'

model_package = sklearn_model.register(
    model_package_group_name=model_package_group_name,
    content_types=['text/csv'],
    response_types=['text/csv'],
    inference_instances=['ml.t2.medium', 'ml.m5.large'],
    transform_instances=['ml.m5.large'],
    approval_status='Approved',
    description='Student Performance - Tuned Random Forest'
)

print(f'Model registered: {model_package.model_package_arn}')

## Summary
- Trained 3 models: Logistic Regression, Decision Tree, Random Forest
- Random Forest best after hyperparameter tuning
- Model artifact uploaded to S3
- Model registered in SageMaker Model Registry